### 1: Завантаження та очищення даних
Завдання: Звантажити та відкрити датасет Individual Household Electric Power Consumption.Здійснити data cleaning.

In [6]:
import pandas as pd
import numpy as np
import timeit

def load_and_clean_data(filepath):
    # Цей датасет використовує ';' як роздільник, а пропуски позначені як '?'
    df = pd.read_csv(filepath, sep=';', na_values=['?'], low_memory=False)
    
    # Видаляємо всі рядки, де є пропущені дані (Data Cleaning)
    df_clean = df.dropna().reset_index(drop=True)
    
    # Перетворюємо стовпці з датою та часом у зручний формат datetime
    # Це займе трохи часу при завантаженні, але прискорить подальшу роботу
    df_clean['Datetime'] = pd.to_datetime(df_clean['Date'] + ' ' + df_clean['Time'], format='%d/%m/%Y %H:%M:%S')
    
    return df_clean

print("Завантажуємо та очищуємо дані... (це може зайняти до хвилини, датасет великий)")
df = load_and_clean_data('household_power_consumption.txt')
print("Дані успішно очищено!")
display(df.head())

Завантажуємо та очищуємо дані... (це може зайняти до хвилини, датасет великий)
Дані успішно очищено!


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,Datetime
0,16/12/2006,17:24:00,4.216,0.418,234.84,18.4,0.0,1.0,17.0,2006-12-16 17:24:00
1,16/12/2006,17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0,2006-12-16 17:25:00
2,16/12/2006,17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0,2006-12-16 17:26:00
3,16/12/2006,17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0,2006-12-16 17:27:00
4,16/12/2006,17:28:00,3.666,0.528,235.68,15.8,0.0,1.0,17.0,2006-12-16 17:28:00


### 2: Вибірка №1 (Загальна потужність > 5 кВт)
Завдання: Обрати всі записи, у яких загальна активна споживана потужність перевищує 5 кВт.Проаналізувати часові витрати за допомогою timeit.

In [7]:
def filter_high_power(dataframe):
    # 'Global_active_power' вимірюється в кіловатах
    return dataframe[dataframe['Global_active_power'] > 5.0]

# 1. Виконуємо функцію для виводу результату
high_power_df = filter_high_power(df)
print(f"Знайдено записів: {len(high_power_df)}")
display(high_power_df.head())

# 2. Використовуємо timeit для профілювання часу виконання
# Створюємо обгортку (wrapper) без аргументів для timeit
wrapper = lambda: filter_high_power(df)

# Запускаємо 10 разів і беремо середнє, щоб отримати точний час
execution_time = timeit.timeit(wrapper, number=10) / 10
print(f"\nСередній час виконання (за 10 запусків): {execution_time:.5f} секунд")

Знайдено записів: 17547


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,Datetime
1,16/12/2006,17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0,2006-12-16 17:25:00
2,16/12/2006,17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0,2006-12-16 17:26:00
3,16/12/2006,17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0,2006-12-16 17:27:00
11,16/12/2006,17:35:00,5.412,0.470,232.78,23.2,0.0,1.0,17.0,2006-12-16 17:35:00
12,16/12/2006,17:36:00,5.224,0.478,232.99,22.4,0.0,1.0,16.0,2006-12-16 17:36:00



Середній час виконання (за 10 запусків): 0.00481 секунд


### 3: Вибірка №2 (Сила струму та порівняння приладів)
[cite_start]Завдання: Обрати всі записи, у яких сила струму лежить в межах 19-20 А, для них виявити ті, у яких пральна машина та холодильних споживають більше, ніж бойлер та кондиціонер.

In [8]:
def filter_appliances(dataframe):
    # Фільтруємо за силою струму (Global_intensity) від 19 до 20 А
    cond_intensity = dataframe['Global_intensity'].between(19, 20)
    
    # Порівнюємо групи споживання: 2 група (пралка/холодильник) > 3 група (бойлер/кондиціонер)
    cond_appliances = dataframe['Sub_metering_2'] > dataframe['Sub_metering_3']
    
    return dataframe[cond_intensity & cond_appliances]

# Виводимо результат
appliances_df = filter_appliances(df)
print(f"Знайдено записів: {len(appliances_df)}")
display(appliances_df.head())

# Вимірюємо час
wrapper_appliances = lambda: filter_appliances(df)
time_appliances = timeit.timeit(wrapper_appliances, number=10) / 10
print(f"Середній час виконання: {time_appliances:.5f} секунд")

Знайдено записів: 2509


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,Datetime
45,16/12/2006,18:09:00,4.464,0.136,234.66,19.0,0.0,37.0,16.0,2006-12-16 18:09:00
460,17/12/2006,01:04:00,4.582,0.258,238.08,19.6,0.0,13.0,0.0,2006-12-17 01:04:00
464,17/12/2006,01:08:00,4.618,0.104,239.61,19.6,0.0,27.0,0.0,2006-12-17 01:08:00
475,17/12/2006,01:19:00,4.636,0.140,237.37,19.4,0.0,36.0,0.0,2006-12-17 01:19:00
476,17/12/2006,01:20:00,4.634,0.152,237.17,19.4,0.0,35.0,0.0,2006-12-17 01:20:00


Середній час виконання: 0.00656 секунд


### 4: Вибірка №3 (Випадкові 500 000 записів)
Завдання: Обрати випадковим чином 500000 записів (без повторів елементів вибірки), для них обчислити середні величини усіх 3-х груп споживання електричної енергії.

In [9]:
def random_sample_means(dataframe):
    # Обираємо 500 000 випадкових записів без повторів (replace=False)
    sample_df = dataframe.sample(n=500000, replace=False)
    
    # Обчислюємо середнє для кожної з трьох груп
    means = sample_df[['Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']].mean()
    return means

# Виводимо результат
means_result = random_sample_means(df)
print("Середні величини груп споживання (кВт-год):")
print(means_result)

# Вимірюємо час
wrapper_sample = lambda: random_sample_means(df)
time_sample = timeit.timeit(wrapper_sample, number=10) / 10
print(f"\nСередній час виконання: {time_sample:.5f} секунд")

Середні величини груп споживання (кВт-год):
Sub_metering_1    1.128158
Sub_metering_2    1.304532
Sub_metering_3    6.432200
dtype: float64

Середній час виконання: 0.11125 секунд


### 5: Складна вечірня вибірка
Завдання: Обрати записи після 18:00 (>6 кВт), де група 2 найбільша. З першої половини обрати кожен 3-й рядок, з другої - кожен 4-й.

In [12]:
def complex_evening_filter(dataframe):
    # 1. Після 18:00
    cond_time = dataframe['Datetime'].dt.hour >= 18
    
    # 2. Понад 6 кВт за хвилину
    cond_power = dataframe['Global_active_power'] > 6.0
    
    # 3. Група 2 є найбільшою (більша за 1 і 3)
    cond_group2 = (dataframe['Sub_metering_2'] > dataframe['Sub_metering_1']) & \
                  (dataframe['Sub_metering_2'] > dataframe['Sub_metering_3'])
                  
    # Застосовуємо всі умови
    filtered_df = dataframe[cond_time & cond_power & cond_group2]
    
    # Якщо нічого не знайдено, повертаємо порожній датафрейм
    if filtered_df.empty:
        return filtered_df
        
    # 4. Ділимо на дві половини
    half_idx = len(filtered_df) // 2
    first_half = filtered_df.iloc[:half_idx]
    second_half = filtered_df.iloc[half_idx:]
    
    # 5. Кожен третій з першої, кожен четвертий з другої (починаємо з 0 індексу)
    res1 = first_half.iloc[::3]
    res2 = second_half.iloc[::4]
    
    return pd.concat([res1, res2])

# Виводимо результат
complex_df = complex_evening_filter(df)
print(f"Знайдено записів після складної фільтрації: {len(complex_df)}")
display(complex_df.head())

# Вимірюємо час
wrapper_complex = lambda: complex_evening_filter(df)
time_complex = timeit.timeit(wrapper_complex, number=10) / 10
print(f"Середній час виконання: {time_complex:.5f} секунд")

Знайдено записів після складної фільтрації: 310


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,Datetime
41,16/12/2006,18:05:00,6.052,0.192,232.93,26.2,0.0,37.0,17.0,2006-12-16 18:05:00
44,16/12/2006,18:08:00,6.308,0.116,232.25,27.0,0.0,36.0,17.0,2006-12-16 18:08:00
17492,28/12/2006,20:58:00,6.386,0.374,236.63,27.0,1.0,36.0,17.0,2006-12-28 20:58:00
17496,28/12/2006,21:02:00,8.088,0.262,235.50,34.4,1.0,72.0,17.0,2006-12-28 21:02:00
17499,28/12/2006,21:05:00,7.230,0.152,235.22,30.6,1.0,73.0,17.0,2006-12-28 21:05:00


Середній час виконання: 0.02776 секунд


### 6: Статистика та кодування
Завдання: Пронормувати та стандартизувати датасет, підрахувати коефіцієнти Пірсона та Спірмена, провести One Hot Encoding.

In [11]:
# 1. Нормалізація (Min-Max) та Стандартизація (Z-score)
# Продемонструємо на колонці Global_active_power вибраного сабсету (high_power_df з кроку 3)
col = 'Global_active_power'
data_subset = high_power_df.copy()

data_subset['Normalized'] = (data_subset[col] - data_subset[col].min()) / (data_subset[col].max() - data_subset[col].min())
data_subset['Standardized'] = (data_subset[col] - data_subset[col].mean()) / data_subset[col].std()

print("Нормалізація та стандартизація:")
display(data_subset[[col, 'Normalized', 'Standardized']].head())

# 2. Коефіцієнти Пірсона та Спірмена (для Global_active_power та Global_intensity)
pearson_val = df['Global_active_power'].corr(df['Global_intensity'], method='pearson')
spearman_val = df['Global_active_power'].corr(df['Global_intensity'], method='spearman')

print(f"\nКоефіцієнт Пірсона: {pearson_val:.4f}")
print(f"Коефіцієнт Спірмена: {spearman_val:.4f}")

# 3. One Hot Encoding
# У датасеті немає явних текстових категорій, тому створимо її з дати (День тижня)
df_ohe = df.copy()
df_ohe['Day_of_week'] = df_ohe['Datetime'].dt.day_name()

# Виконуємо кодування
ohe_result = pd.get_dummies(df_ohe['Day_of_week']).astype(int)
print("\nOne Hot Encoding для днів тижня:")
display(ohe_result.head())

Нормалізація та стандартизація:


,Global_active_power,Normalized,Standardized
1,5.360,0.058497,-0.632864
2,5.374,0.060784,-0.614899
3,5.388,0.063072,-0.596935
11,5.412,0.066993,-0.566138
12,5.224,0.036275,-0.807376



Коефіцієнт Пірсона: 0.9989
Коефіцієнт Спірмена: 0.9954

One Hot Encoding для днів тижня:


,Friday,Monday,Saturday,Sunday,Thursday,Tuesday,Wednesday
0,0,0,1,0,0,0,0
1,0,0,1,0,0,0,0
2,0,0,1,0,0,0,0
3,0,0,1,0,0,0,0
4,0,0,1,0,0,0,0
